習題一：使用random模組，產生出一個介於0到1之間的小數$r$，使用while迴圈，計算出$r$的前二十位二進位表示$r = 0.b_1b_2b_3\cdots {b_{20}}_2$。輸出格式如附圖Exercise20261015.png

In [8]:
import random

# 產生一個介於 0 到 1 之間的小數 r
r = random.random()
print(f"產生的十進位小數 r = {r}")

# 初始化變數
temp = r
bits = []
count = 0

# 使用 while 迴圈計算前二十位二進位值
while count < 20:
    temp *= 2
    bit = int(temp)
    bits.append(str(bit))
    temp -= bit
    count += 1

# 組合二進位表示字串
binary_str = "0." + "".join(bits)
print(f"r 的前二十位二進位表示法 = {binary_str}")

產生的十進位小數 r = 0.7754017276646779
r 的前二十位二進位表示法 = 0.11000110100000001011


Python 的 `int` 型別採用 Arbitrary-Precision Integer（任意精度整數）設計，能夠精確表示任意大小的整數，其實作方式可參考 CPython 的 `PyLongObject`。與 C/C++ 的 `int` 或 `long long` 不同，Python 的整數不受固定 32 位元或64 位元限制，而是會依據數值大小自動配置更多記憶體，因此只要記憶體足夠，便能表示極大的整數。例如：
```python
n = 10**68
print(n)
```
上述程式可正常執行，表示 Python 能夠精確儲存$10^{68}$，即中文數字中的「無量大數」。


可利用下列程式觀察不同大小整數所需的位元數（bit）與記憶體空間：

```python
import sys
n = 10**68
print(f"bit_length = {n.bit_length()}")
print(f"memory = {sys.getsizeof(n)} bytes")
```

並查閱官方文件，了解：

- `bit_length()`
- `sys.getsizeof()`
- `PyLongObject`

三者之間的關係。

習題二：輸入一個正整數 \(n\)，其中$1 \le n \le 10^{68}$，請將 \(n\) 轉換為中文大寫數字表示法。需支援下列數字單位：萬、億、兆、京、垓、秭、穰、溝、澗、正、載、極、恆河沙、阿僧祇、那由他、不可思議、無量大數。比如輸入31415926，輸出參仟壹佰肆拾壹萬伍仟玖佰貳拾陸。

提示：思考利用字典，建立「數字」與「中文大寫數字」的對應關係。

In [3]:
def number_to_chinese_uppercase(n_str):
    # 確保輸入是字串且只包含數字
    n_str = str(n_str).strip()
    if not n_str.isdigit():
        return "輸入錯誤，請輸入正整數"

    # 中文大寫數字對應
    digits_map = {
        '0': '零', '1': '壹', '2': '貳', '3': '參', '4': '肆',
        '5': '伍', '6': '陸', '7': '柒', '8': '捌', '9': '玖'
    }

    # 四位數內部的單位
    sub_units = ['', '拾', '佰', '仟']

    # 每四位數（萬進位）的大單位對應
    # 10^4 = 萬, 10^8 = 億, 10^12 = 兆 ... 10^68 = 無量大數
    units = [
        '', '萬', '億', '兆', '京', '垓', '秭', '穰', '溝', '澗',
        '正', '載', '極', '恆河沙', '阿僧祇', '那由他', '不可思議', '無量大數'
    ]

    # 將數字左邊補零，使其長度為 4 的倍數，便於每 4 位一組處理
    length = len(n_str)
    remainder = length % 4
    if remainder != 0:
        n_str = n_str.zfill(length + (4 - remainder))

    # 每 4 位切成一組
    chunks = [n_str[i:i+4] for i in range(0, len(n_str), 4)]
    chunks.reverse() # 從個位數方向往高位處理

    result_parts = []

    for idx, chunk in enumerate(chunks):
        # 處理這 4 位數
        chunk_str = ""
        zero_flag = False

        for sub_idx, char in enumerate(chunk):
            power = 3 - sub_idx # 3對應仟位，2對應佰位，1對應拾位，0對應個位
            digit = char

            if digit != '0':
                if zero_flag:
                    chunk_str += '零'
                    zero_flag = False
                chunk_str += digits_map[digit] + sub_units[power]
            else:
                # 只有在後面還有非零數字時，才標記需要補「零」
                if chunk_str != "":
                    zero_flag = True

        if chunk_str != "":
            # 如果此 4 位數不全為零，加上該組的大單位（萬、億等）
            if idx < len(units):
                chunk_str += units[idx]
            result_parts.append(chunk_str)

    if not result_parts:
        return "零"

    result_parts.reverse()
    return "".join(result_parts)

# 讓使用者動態輸入數字
user_input = input("請輸入一個正整數（最高可達 10^68）：")
result = number_to_chinese_uppercase(user_input)
print(f"\n轉換結果：\n{result}")

請輸入一個正整數（最高可達 10^68）：123456789

轉換結果：
壹億貳仟參佰肆拾伍萬陸仟柒佰捌拾玖


習題三：請勿使用for、range，而是利用iterable、iter()、next()、while、StopIteration來設計九九乘法表。

In [4]:
# 定義 1 到 9 的串列做為可迭代對象
numbers_i = [1, 2, 3, 4, 5, 6, 7, 8, 9]

# 建立外層迭代器
iter_i = iter(numbers_i)

while True:
    try:
        # 取得外層數值 i
        i = next(iter_i)

        # 每次外層循環，都為內層建立一個 1 到 9 的迭代器
        numbers_j = [1, 2, 3, 4, 5, 6, 7, 8, 9]
        iter_j = iter(numbers_j)

        row_results = []
        while True:
            try:
                # 取得內層數值 j
                j = next(iter_j)
                # 格式化乘法結果，例如 "2x3=06"
                row_results.append(f"{i}x{j}={i*j:02d}")
            except StopIteration:
                # 當內層迭代結束時，跳出內層迴圈
                break

        # 印出該列（i 乘上所有 j 的結果）
        print("  ".join(row_results))

    except StopIteration:
        # 當外層迭代結束時，跳出外層迴圈
        break

1x1=01  1x2=02  1x3=03  1x4=04  1x5=05  1x6=06  1x7=07  1x8=08  1x9=09
2x1=02  2x2=04  2x3=06  2x4=08  2x5=10  2x6=12  2x7=14  2x8=16  2x9=18
3x1=03  3x2=06  3x3=09  3x4=12  3x5=15  3x6=18  3x7=21  3x8=24  3x9=27
4x1=04  4x2=08  4x3=12  4x4=16  4x5=20  4x6=24  4x7=28  4x8=32  4x9=36
5x1=05  5x2=10  5x3=15  5x4=20  5x5=25  5x6=30  5x7=35  5x8=40  5x9=45
6x1=06  6x2=12  6x3=18  6x4=24  6x5=30  6x6=36  6x7=42  6x8=48  6x9=54
7x1=07  7x2=14  7x3=21  7x4=28  7x5=35  7x6=42  7x7=49  7x8=56  7x9=63
8x1=08  8x2=16  8x3=24  8x4=32  8x5=40  8x6=48  8x7=56  8x8=64  8x9=72
9x1=09  9x2=18  9x3=27  9x4=36  9x5=45  9x6=54  9x7=63  9x8=72  9x9=81
